# ✂️ Chunking Techniques: Breaking Documents into Bite-Sized Pieces

## 🎯 What You'll Learn

- Why we need to split documents into smaller pieces
- Fixed-size chunking (simplest approach)
- Overlapping chunks (and why they help)
- Sentence-based chunking
- Semantic chunking (the smart approach)
- Recursive chunking (LangChain's approach)
- How to choose the right strategy

**Prerequisites:** Understanding of RAG ([Notebook 01](01_what_is_rag.ipynb))

---

## 📚 Why Do We Need Chunking?

Imagine you're studying for a big test. Would you try to memorize a **whole textbook** as one giant piece? Of course not! You'd break it into chapters, sections, and key concepts. That's exactly what **chunking** does for RAG.

Here's why it matters:

- **LLMs have limited context windows** — think of it like having limited desk space. You can only fit so many papers on your desk at once.
- **Smaller chunks = more precise retrieval** — it's easier to find the exact answer when your notes are organized.
- **A 100-page document won't fit in one embedding effectively** — trying to summarize a whole book in one sentence loses all the details.
- **Each chunk gets its own embedding** — like giving each section its own GPS coordinate in meaning-space.
- **If a chunk is too big, the embedding becomes a blurry average** of many topics — like a photo that's out of focus.

💡 **Chunking is like creating a detailed table of contents** — it helps the system quickly find exactly what it needs!

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# === Left side: One huge document → one blurry embedding ===
ax1.set_title('❌ Without Chunking', fontsize=14, fontweight='bold', color='#c0392b')
ax1.set_xlim(0, 10)
ax1.set_ylim(0, 10)
ax1.axis('off')

# Big document blob
big_doc = patches.FancyBboxPatch((0.5, 4), 4, 5, boxstyle='round,pad=0.3',
                                  facecolor='#e74c3c', alpha=0.3, edgecolor='#c0392b', linewidth=2)
ax1.add_patch(big_doc)
ax1.text(2.5, 6.5, '📄 Entire\nDocument\n(100 pages)', ha='center', va='center', fontsize=11, fontweight='bold')

# Arrow
ax1.annotate('', xy=(7, 6.5), xytext=(5, 6.5),
             arrowprops=dict(arrowstyle='->', lw=2, color='gray'))

# Blurry embedding
blurry = patches.Circle((8, 6.5), 1.2, facecolor='#e74c3c', alpha=0.15, edgecolor='#c0392b',
                         linewidth=2, linestyle='--')
ax1.add_patch(blurry)
ax1.text(8, 6.5, '🔴\nBlurry\nEmbedding', ha='center', va='center', fontsize=9, color='#c0392b')

ax1.text(5, 2.5, '"What is this document about?"\n→ A little bit of everything...\n→ Not useful for finding specifics!',
         ha='center', va='center', fontsize=10, style='italic',
         bbox=dict(boxstyle='round,pad=0.5', facecolor='#fadbd8', edgecolor='#c0392b'))

# === Right side: Split into chunks → precise embeddings ===
ax2.set_title('✅ With Chunking', fontsize=14, fontweight='bold', color='#27ae60')
ax2.set_xlim(0, 10)
ax2.set_ylim(0, 10)
ax2.axis('off')

# Multiple small chunks
colors = ['#3498db', '#2ecc71', '#f39c12', '#9b59b6']
labels = ['Chunk 1:\nHistory', 'Chunk 2:\nScience', 'Chunk 3:\nMath', 'Chunk 4:\nArt']
for i, (color, label) in enumerate(zip(colors, labels)):
    y = 8.5 - i * 1.8
    chunk = patches.FancyBboxPatch((0.3, y - 0.5), 2.5, 1.2, boxstyle='round,pad=0.2',
                                   facecolor=color, alpha=0.3, edgecolor=color, linewidth=2)
    ax2.add_patch(chunk)
    ax2.text(1.55, y + 0.1, label, ha='center', va='center', fontsize=9, fontweight='bold')

    # Arrow
    ax2.annotate('', xy=(5.5, y + 0.1), xytext=(3, y + 0.1),
                 arrowprops=dict(arrowstyle='->', lw=1.5, color='gray'))

    # Precise embedding dot
    emb = patches.Circle((6.5 + (i % 2) * 1.5, y + 0.1), 0.4, facecolor=color, alpha=0.5,
                          edgecolor=color, linewidth=2)
    ax2.add_patch(emb)
    ax2.text(6.5 + (i % 2) * 1.5, y + 0.1, '🎯', ha='center', va='center', fontsize=10)

ax2.text(7.2, 1.5, 'Each chunk has a\nPRECISE embedding!\n→ Easy to find exactly\n   what you need!',
         ha='center', va='center', fontsize=10, style='italic',
         bbox=dict(boxstyle='round,pad=0.5', facecolor='#d5f5e3', edgecolor='#27ae60'))

plt.suptitle('Why Chunking Matters for RAG', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---

## ✂️ Method 1: Fixed-Size Chunking

The simplest approach: **split every N characters**, no matter what.

Think of it like **cutting a pizza into equal slices regardless of where the toppings are** 🍕 — you might cut right through a pepperoni!

| Pros | Cons |
|------|------|
| ✅ Simple to implement | ❌ Can cut sentences in half! |
| ✅ Predictable chunk sizes | ❌ May split related information |
| ✅ Fast processing | ❌ Chunks may start/end mid-thought |

In [ ]:
def fixed_size_chunking(text, chunk_size=200):
    """Split text into fixed-size chunks of `chunk_size` characters."""
    chunks = []
    for i in range(0, len(text), chunk_size):
        chunks.append(text[i:i + chunk_size])
    return chunks


# Sample text about space
sample_text = (
    "The solar system is a fascinating place that has captivated human imagination for thousands of years. "
    "Our Sun, a medium-sized star, sits at the center and provides light and warmth to eight planets. "
    "Mercury is the closest planet to the Sun and has extreme temperature swings. "
    "Venus is often called Earth's twin because of its similar size, but its atmosphere is incredibly hostile. "
    "Earth is the only known planet to support life, with its perfect distance from the Sun and protective atmosphere. "
    "Mars, the Red Planet, has been the target of numerous space missions searching for signs of past life. "
    "Jupiter is the largest planet, a gas giant with a famous Great Red Spot storm that has raged for centuries. "
    "Saturn is known for its beautiful ring system made of ice and rock particles. "
    "Uranus and Neptune are the ice giants of the outer solar system, mysterious and far from the Sun."
)

print(f"📄 Original text length: {len(sample_text)} characters")
print("=" * 60)

chunks = fixed_size_chunking(sample_text, chunk_size=200)

for i, chunk in enumerate(chunks):
    # Check if chunk starts or ends mid-sentence
    starts_mid = (i > 0 and not chunk[0].isupper() and chunk[0] != ' ')
    ends_mid = (i < len(chunks) - 1 and not chunk.rstrip().endswith(('.', '!', '?')))

    warning = ""
    if starts_mid:
        warning += " ⚠️ Starts mid-sentence!"
    if ends_mid:
        warning += " ⚠️ Ends mid-sentence!"

    print(f"\n📦 Chunk {i + 1} ({len(chunk)} chars){warning}")
    print(f"   \"{chunk}\"")

print("\n" + "=" * 60)
print(f"Total chunks: {len(chunks)}")
print("\n⚠️ Notice how some chunks cut right through sentences!")
print("   This means a search about Venus might miss half the information.")

---

## 🔗 Method 2: Overlapping Chunks

**Problem:** With fixed-size chunking, information at chunk boundaries gets lost. If a sentence is split between two chunks, neither chunk has the full thought.

**Solution:** Make chunks **overlap**! Each chunk shares some text with the previous one.

🏠 **Analogy:** Think of **shingles on a roof** — they overlap so that rain can't get through the cracks. Similarly, overlapping chunks ensure no information falls through the gaps!

**Typical overlap:** 10-20% of chunk size (e.g., chunk_size=200, overlap=40)

In [ ]:
def overlapping_chunking(text, chunk_size=200, overlap=50):
    """Split text into overlapping chunks."""
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start = end - overlap  # Step back by `overlap` characters
    return chunks


print("🔗 Overlapping Chunking (chunk_size=200, overlap=50)")
print("=" * 60)

overlap_chunks = overlapping_chunking(sample_text, chunk_size=200, overlap=50)

for i, chunk in enumerate(overlap_chunks):
    print(f"\n📦 Chunk {i + 1} ({len(chunk)} chars)")
    print(f"   \"{chunk}\"")

    # Show overlap with next chunk
    if i < len(overlap_chunks) - 1:
        overlap_text = chunk[-50:] if len(chunk) >= 50 else chunk
        print(f"   🔗 Overlap region (last 50 chars): \"{overlap_text}\"")

print("\n" + "=" * 60)
print(f"Total chunks: {len(overlap_chunks)} (vs {len(chunks)} without overlap)")
print("\n✅ Now boundary information is preserved in overlapping regions!")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 4))

text_len = len(sample_text)

# === Top: Fixed chunks (with gaps highlighted) ===
ax1.set_title('❌ Fixed-Size Chunks — Information Lost at Boundaries', fontsize=12, fontweight='bold')
ax1.set_xlim(0, text_len)
ax1.set_ylim(0, 2)
ax1.axis('off')

colors_fixed = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12', '#9b59b6']
chunk_size = 200
for i, start in enumerate(range(0, text_len, chunk_size)):
    end = min(start + chunk_size, text_len)
    color = colors_fixed[i % len(colors_fixed)]
    rect = patches.FancyBboxPatch((start + 2, 0.4), end - start - 4, 1.2,
                                  boxstyle='round,pad=0.05',
                                  facecolor=color, alpha=0.4, edgecolor=color, linewidth=2)
    ax1.add_patch(rect)
    ax1.text((start + end) / 2, 1.0, f'Chunk {i + 1}', ha='center', va='center',
             fontsize=9, fontweight='bold')

    # Mark boundary gaps
    if i > 0:
        ax1.axvline(x=start, color='red', linestyle='--', linewidth=1.5, alpha=0.7)
        ax1.text(start, 0.15, '⚡', ha='center', fontsize=10)

# === Bottom: Overlapping chunks ===
ax2.set_title('✅ Overlapping Chunks — Boundaries Are Covered', fontsize=12, fontweight='bold')
ax2.set_xlim(0, text_len)
ax2.set_ylim(0, 2)
ax2.axis('off')

overlap = 50
start = 0
i = 0
while start < text_len:
    end = min(start + chunk_size, text_len)
    color = colors_fixed[i % len(colors_fixed)]

    # Main chunk body
    rect = patches.FancyBboxPatch((start + 2, 0.4), end - start - 4, 1.2,
                                  boxstyle='round,pad=0.05',
                                  facecolor=color, alpha=0.3, edgecolor=color, linewidth=2)
    ax2.add_patch(rect)
    ax2.text((start + end) / 2, 1.0, f'Chunk {i + 1}', ha='center', va='center',
             fontsize=9, fontweight='bold')

    # Highlight overlap region
    if i > 0:
        overlap_start = start
        overlap_end = start + overlap
        overlap_rect = patches.FancyBboxPatch((overlap_start, 0.3), overlap, 1.4,
                                              boxstyle='round,pad=0.02',
                                              facecolor='#f1c40f', alpha=0.4,
                                              edgecolor='#f39c12', linewidth=1.5,
                                              linestyle='--')
        ax2.add_patch(overlap_rect)
        ax2.text((overlap_start + overlap_end) / 2, 0.12, '🔗', ha='center', fontsize=8)

    start = end - overlap
    i += 1

plt.tight_layout()
plt.show()

---

## 📝 Method 3: Sentence-Based Chunking

Instead of cutting at a fixed number of characters, **respect sentence boundaries**!

The idea is simple:
- Split the text into sentences
- Group complete sentences together until you reach the target chunk size
- **Never cut a thought in half** ✂️❌

Trade-off: chunk sizes will vary more, but each chunk contains **complete thoughts**.

In [ ]:
def sentence_chunking(text, max_chunk_size=300):
    """Split text into chunks that respect sentence boundaries."""
    # Simple sentence splitting (handles . ! ?)
    sentences = text.replace('! ', '!|').replace('? ', '?|').replace('. ', '.|').split('|')
    chunks = []
    current = ''
    for s in sentences:
        if len(current) + len(s) > max_chunk_size and current:
            chunks.append(current.strip())
            current = s
        else:
            current += ' ' + s
    if current.strip():
        chunks.append(current.strip())
    return chunks


print("📝 Sentence-Based Chunking (max_chunk_size=300)")
print("=" * 60)

sent_chunks = sentence_chunking(sample_text, max_chunk_size=300)

for i, chunk in enumerate(sent_chunks):
    ends_clean = chunk.rstrip().endswith(('.', '!', '?'))
    status = "✅ Complete sentences" if ends_clean else "⚠️ Partial sentence"
    print(f"\n📦 Chunk {i + 1} ({len(chunk)} chars) — {status}")
    print(f"   \"{chunk}\"")

print("\n" + "=" * 60)
print("\n🔍 Comparison with Fixed-Size Chunking:")
print(f"   Fixed-size chunks: {len(chunks)} chunks (many cut mid-sentence)")
print(f"   Sentence chunks:   {len(sent_chunks)} chunks (all complete sentences!)")
print("\n✅ Every chunk contains complete, coherent sentences!")

---

## 🧠 Method 4: Semantic Chunking

The **smartest** approach: split based on **MEANING**, not size!

🎵 **Analogy:** Imagine a DJ who changes songs **when the mood of the party shifts**, not on a fixed timer. Semantic chunking detects when the **topic changes** and creates a new chunk there.

**How it works:**
1. Split text into sentences
2. Compare each pair of adjacent sentences using **similarity**
3. When similarity **drops significantly**, that's a topic boundary!
4. Cut the chunk there

This uses **embeddings** to detect topic changes — each sentence gets a vector, and we measure how "close" neighboring sentences are in meaning-space.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Simulated similarity scores between adjacent sentence pairs
# (In real life, these come from comparing embedding vectors)
sentence_labels = [
    'S1-S2\n(both about\nsolar system)', 
    'S2-S3\n(Sun to\nMercury)',
    'S3-S4\n(Mercury to\nVenus)', 
    'S4-S5\n(Venus to\nEarth)',
    'S5-S6\n(Earth to\nMars)',
    'S6-S7\n(Mars to\nJupiter)', 
    'S7-S8\n(Jupiter to\nSaturn)',
    'S8-S9\n(Saturn to\nice giants)'
]

# Fake but illustrative: high similarity within topics, drops at boundaries
similarities = [0.85, 0.35, 0.72, 0.68, 0.65, 0.25, 0.78, 0.30]
threshold = 0.4

fig, ax = plt.subplots(figsize=(14, 5))

x = np.arange(len(similarities))
colors = ['#2ecc71' if s >= threshold else '#e74c3c' for s in similarities]

bars = ax.bar(x, similarities, color=colors, alpha=0.7, edgecolor='black', linewidth=1)
ax.axhline(y=threshold, color='red', linestyle='--', linewidth=2, label=f'Threshold = {threshold}')

# Annotate chunk boundaries
for i, sim in enumerate(similarities):
    if sim < threshold:
        ax.annotate('✂️ CHUNK\nBOUNDARY', xy=(i, sim), xytext=(i, sim + 0.15),
                    fontsize=10, fontweight='bold', color='#c0392b', ha='center',
                    arrowprops=dict(arrowstyle='->', color='#c0392b', lw=2))
    else:
        ax.text(i, sim + 0.03, f'{sim:.2f}', ha='center', fontsize=9, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(sentence_labels, fontsize=8)
ax.set_ylabel('Similarity Score', fontsize=12)
ax.set_xlabel('Adjacent Sentence Pairs', fontsize=12)
ax.set_title('🧠 Semantic Chunking: Detecting Topic Boundaries\n'
             'Low similarity = topic change = chunk boundary!', fontsize=14, fontweight='bold')
ax.set_ylim(0, 1.1)
ax.legend(fontsize=11, loc='upper right')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
def jaccard_similarity(s1, s2):
    """Simple word-overlap similarity (no ML needed!)."""
    w1 = set(s1.lower().split())
    w2 = set(s2.lower().split())
    if not (w1 | w2):
        return 0
    return len(w1 & w2) / len(w1 | w2)


def semantic_chunking(text, threshold=0.1):
    """Split text into chunks based on semantic similarity between sentences."""
    sentences = text.replace('! ', '!|').replace('? ', '?|').replace('. ', '.|').split('|')
    sentences = [s.strip() for s in sentences if s.strip()]
    
    if not sentences:
        return []
    
    chunks = []
    current_chunk = [sentences[0]]
    
    for i in range(1, len(sentences)):
        sim = jaccard_similarity(sentences[i - 1], sentences[i])
        if sim < threshold:
            # Topic changed! Start a new chunk
            chunks.append(' '.join(current_chunk))
            current_chunk = [sentences[i]]
        else:
            # Same topic, continue current chunk
            current_chunk.append(sentences[i])
    
    if current_chunk:
        chunks.append(' '.join(current_chunk))
    
    return chunks


# A text with clear topic changes
multi_topic_text = (
    "Photosynthesis is the process by which plants convert sunlight into energy. "
    "Plants use chlorophyll in their leaves to absorb light from the sun. "
    "The light energy is used to convert carbon dioxide and water into glucose and oxygen. "
    "Basketball was invented by James Naismith in 1891 in Springfield, Massachusetts. "
    "The game is played with two teams of five players each on a rectangular court. "
    "Players score by shooting a ball through the opponent's hoop mounted ten feet high. "
    "Python is a popular programming language created by Guido van Rossum. "
    "It emphasizes code readability and allows programmers to express concepts in fewer lines. "
    "Python is widely used in web development, data science, and artificial intelligence."
)

print("🧠 Semantic Chunking Demo")
print("=" * 60)
print("\nInput text has 3 topics: 🌱 Photosynthesis, 🏀 Basketball, 🐍 Python")
print()

# Show sentence-by-sentence similarities
sentences = multi_topic_text.replace('! ', '!|').replace('? ', '?|').replace('. ', '.|').split('|')
sentences = [s.strip() for s in sentences if s.strip()]

print("📊 Similarity between adjacent sentences:")
for i in range(1, len(sentences)):
    sim = jaccard_similarity(sentences[i - 1], sentences[i])
    indicator = "🔗 Same topic" if sim >= 0.1 else "✂️ TOPIC CHANGE"
    print(f"   S{i}→S{i + 1}: similarity = {sim:.3f}  {indicator}")

print("\n" + "-" * 60)
print("\n📦 Resulting chunks:")
sem_chunks = semantic_chunking(multi_topic_text, threshold=0.1)

topic_emojis = ['🌱', '🏀', '🐍', '📌', '📌']
for i, chunk in enumerate(sem_chunks):
    emoji = topic_emojis[i] if i < len(topic_emojis) else '📌'
    print(f"\n{emoji} Chunk {i + 1} ({len(chunk)} chars):")
    print(f"   \"{chunk}\"")

print(f"\n✅ Semantic chunking correctly identified {len(sem_chunks)} topic groups!")

---

## 🪆 Method 5: Recursive Chunking

This is the approach used by **LangChain's `RecursiveCharacterTextSplitter`** — one of the most popular chunking tools.

The idea: try the **most meaningful separators first**, and only fall back to less meaningful ones if needed:

1. First, try splitting on `\n\n` (paragraph breaks)
2. If chunks are still too big, split on `\n` (line breaks)
3. Still too big? Split on `. ` (sentences)
4. Last resort: split on ` ` (spaces/words)

🗄️ **Analogy:** It's like **organizing a messy closet**: first sort by category (pants, shirts, shoes), then by color within each category, then by size within each color.

In [ ]:
def recursive_chunking(text, chunk_size=200, separators=None):
    """Split text recursively using a hierarchy of separators."""
    if separators is None:
        separators = ['\n\n', '\n', '. ', ' ']
    
    # Base case: text fits in one chunk
    if len(text) <= chunk_size:
        return [text.strip()] if text.strip() else []
    
    # Try each separator in order (most meaningful first)
    for sep in separators:
        if sep in text:
            parts = text.split(sep)
            chunks = []
            current = ''
            
            for part in parts:
                # Add separator back (except for spaces)
                test_addition = part if sep == ' ' else part + sep.rstrip()
                
                if len(current) + len(test_addition) + len(sep) <= chunk_size:
                    current = current + sep + part if current else part
                else:
                    if current.strip():
                        chunks.append(current.strip())
                    current = part
            
            if current.strip():
                chunks.append(current.strip())
            
            # Recursively split any chunks that are still too big
            remaining_seps = separators[separators.index(sep) + 1:]
            final_chunks = []
            for chunk in chunks:
                if len(chunk) > chunk_size and remaining_seps:
                    final_chunks.extend(recursive_chunking(chunk, chunk_size, remaining_seps))
                else:
                    final_chunks.append(chunk)
            
            return final_chunks
    
    # Fallback: hard split (shouldn't reach here often)
    return fixed_size_chunking(text, chunk_size)


# Multi-paragraph text with clear structure
structured_text = """Introduction to Machine Learning

Machine learning is a subset of artificial intelligence that enables systems to learn from data. Instead of being explicitly programmed, these systems identify patterns and make decisions with minimal human intervention.

Supervised Learning

In supervised learning, the algorithm learns from labeled training data. Each example in the training set has an input and a desired output. The algorithm learns a function that maps inputs to outputs. Common examples include email spam detection and image classification.

Unsupervised Learning

Unsupervised learning works with data that has no labels. The algorithm tries to find hidden patterns or structures in the data. Clustering and dimensionality reduction are popular unsupervised techniques. Customer segmentation is a real-world application.

Reinforcement Learning

Reinforcement learning involves an agent that learns by interacting with an environment. The agent receives rewards or penalties based on its actions. Over time, it learns a strategy that maximizes cumulative reward. Game playing AI and robotics use this approach."""

print("🪆 Recursive Chunking Demo")
print("=" * 60)
print(f"\n📄 Original text: {len(structured_text)} characters")
print(f"   Contains {structured_text.count(chr(10) + chr(10))} paragraph breaks (\\n\\n)")

rec_chunks = recursive_chunking(structured_text, chunk_size=250)

print(f"\n📦 Result: {len(rec_chunks)} chunks\n")

for i, chunk in enumerate(rec_chunks):
    print(f"--- Chunk {i + 1} ({len(chunk)} chars) ---")
    print(chunk)
    print()

print("✅ Notice how recursive chunking respects paragraph structure!")
print("   Each chunk tends to cover one coherent topic.")

---

## 📏 How Chunk Size Affects RAG Quality

Choosing the right chunk size is like the story of **Goldilocks and the Three Bears** 🐻:

- **Too small** (< 100 tokens): Chunks lack enough context. Searching for "What causes rain?" might return just "water evaporates" without the full explanation.
- **Too large** (> 1000 tokens): Chunks are too general. The embedding becomes a blurry average of many ideas.
- **Just right** (100-500 tokens): Chunks have enough context to be meaningful, but are focused enough to be found precisely.

The **sweet spot** is typically **200-500 tokens** for most RAG applications.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Chunk sizes and synthetic retrieval quality scores
chunk_sizes = [50, 100, 200, 500, 1000, 2000]
quality_scores = [0.35, 0.62, 0.88, 0.92, 0.65, 0.38]

fig, ax = plt.subplots(figsize=(12, 6))

# Color bars based on quality
colors = []
for score in quality_scores:
    if score >= 0.8:
        colors.append('#2ecc71')  # Green - good
    elif score >= 0.6:
        colors.append('#f39c12')  # Orange - okay
    else:
        colors.append('#e74c3c')  # Red - bad

bars = ax.bar([str(s) for s in chunk_sizes], quality_scores, color=colors, 
              alpha=0.8, edgecolor='black', linewidth=1.5, width=0.6)

# Add value labels on bars
for bar, score in zip(bars, quality_scores):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
            f'{score:.0%}', ha='center', va='bottom', fontsize=12, fontweight='bold')

# Zone annotations
ax.annotate('❄️ Too Small!\nLacks context', xy=(0, 0.35), xytext=(0, 0.55),
            fontsize=11, ha='center', color='#c0392b', fontweight='bold',
            arrowprops=dict(arrowstyle='->', color='#c0392b', lw=2))

ax.annotate('🔥 Too Big!\nToo general', xy=(5, 0.38), xytext=(5, 0.55),
            fontsize=11, ha='center', color='#c0392b', fontweight='bold',
            arrowprops=dict(arrowstyle='->', color='#c0392b', lw=2))

# Highlight the sweet spot
ax.axvspan(1.5, 3.5, alpha=0.1, color='green')
ax.text(2.5, 1.0, '🎯 SWEET SPOT\n(200-500 tokens)', ha='center', va='center',
        fontsize=14, fontweight='bold', color='#27ae60',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='#27ae60', alpha=0.9))

ax.set_xlabel('Chunk Size (tokens)', fontsize=13, fontweight='bold')
ax.set_ylabel('Retrieval Quality', fontsize=13, fontweight='bold')
ax.set_title('📏 The Goldilocks Zone: Finding the Right Chunk Size', fontsize=15, fontweight='bold')
ax.set_ylim(0, 1.15)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

---

## 📊 Comparison Table

| Method | Complexity | Pros | Cons | Best For |
|--------|-----------|------|------|----------|
| **Fixed-size** | ⭐ | Simple, fast, predictable | Cuts sentences in half | Quick prototypes |
| **Overlapping** | ⭐⭐ | Better boundary coverage | Some redundancy | Most use cases |
| **Sentence** | ⭐⭐ | Clean sentence boundaries | Varying chunk sizes | Clean, well-written text |
| **Semantic** | ⭐⭐⭐ | Topic-coherent chunks | Needs embeddings/similarity | High-quality retrieval |
| **Recursive** | ⭐⭐⭐ | Respects document structure | More complex logic | Structured documents |

## 💡 Practical Tips for Choosing Your Chunking Strategy

1. **Start simple:** Begin with **overlapping fixed-size chunks** (200-500 tokens, 50-100 token overlap). This works well for most cases.

2. **Structured documents?** Use **recursive chunking** — it respects headers, paragraphs, and sections.

3. **Need high quality?** Try **semantic chunking** — it keeps related information together.

4. **Always test with YOUR data** — different documents benefit from different strategies!

5. **Consider your LLM's context window** — if your LLM can handle 8K tokens, you can use slightly larger chunks.

---

## 🎯 Key Takeaways

1. **Chunking breaks documents into searchable, embeddable pieces** — it's a critical step in any RAG pipeline.

2. **Fixed-size chunking** is the simplest but can cut sentences and thoughts in half.

3. **Overlapping chunks** prevent information loss at boundaries — like shingles on a roof.

4. **Sentence-based chunking** respects natural language boundaries — never cuts a thought in half.

5. **Semantic chunking** splits by topic changes — the smartest approach for quality retrieval.

6. **Recursive chunking** respects document structure (paragraphs → sentences → words).

7. **Chunk size of 200-500 tokens** is usually the sweet spot for RAG applications.

---

## 🤔 Test Your Understanding

**Question 1:** Why can't we just embed an entire document as one big chunk?

<details>
<summary>👉 Click to reveal answer</summary>

When you embed a huge document as one chunk, the embedding becomes a **blurry average** of all the topics in the document. It's like trying to describe a 300-page book in a single sentence — you lose all the specific details. Smaller chunks produce **focused embeddings** that can be precisely matched to specific queries.

</details>

---

**Question 2:** What problem does overlapping chunking solve that fixed-size chunking doesn't?

<details>
<summary>👉 Click to reveal answer</summary>

Overlapping chunking solves the **boundary information loss** problem. With fixed-size chunking, a sentence split between two chunks means neither chunk has the complete thought. With overlap, the shared region ensures that **boundary information appears in both chunks**, so it can always be found during retrieval.

</details>

---

**Question 3:** When would you choose semantic chunking over sentence-based chunking?

<details>
<summary>👉 Click to reveal answer</summary>

Choose **semantic chunking** when your text covers **multiple topics** and you want each chunk to be about **one coherent topic**. Sentence-based chunking groups sentences by size, so a chunk might contain sentences about two different topics if they happen to be adjacent. Semantic chunking detects **topic boundaries** and splits there, keeping each chunk topically focused.

</details>

---

**Question 4:** Why is 200-500 tokens considered the "sweet spot" for chunk size?

<details>
<summary>👉 Click to reveal answer</summary>

This range balances two competing needs: **enough context** to be meaningful (a 50-token chunk might just say "water evaporates" without explaining why) and **enough focus** to produce a precise embedding (a 2000-token chunk covers too many ideas). At 200-500 tokens, chunks typically contain **one complete idea or paragraph**, making them ideal for both embedding quality and retrieval precision.

</details>

---

**Question 5:** In recursive chunking, why do we try paragraph separators before sentence separators?

<details>
<summary>👉 Click to reveal answer</summary>

Paragraph breaks (`\n\n`) are the **most meaningful** separators because they typically indicate a **topic or section change**. By splitting on paragraphs first, we preserve the document's intended structure. We only fall back to sentence or word splitting when a single paragraph is still too large for our target chunk size. This preserves the **hierarchy of meaning** in the document.

</details>

---

## 🚀 What's Next?

Now that you know how to break documents into chunks, the next step is learning **where to store them** and **how to search them efficiently**!

👉 [**Notebook 03: Vector Databases — Storing and Searching Your Chunks**](03_vector_databases.ipynb)

In the next notebook, you'll learn:
- What vector databases are and why we need them
- How similarity search works at scale
- Popular vector database options (FAISS, Chroma, Pinecone)
- How to build a working vector store from scratch